In [1]:
import kagglehub
import pandas as pd
import numpy as np

d:\Projetos\Machine-learning-exercises\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 01. Leitura do Dataset

In [2]:
# Download latest version
path = kagglehub.dataset_download("ranafayezz/titanic-cleaned")

df_train = pd.read_csv(path+"/train_data.csv")
df_test = pd.read_csv(path+"/test_data.csv")

# 02. Construção do Exercício

In [3]:
# Vou gerar uma variável qualitativa ordinal a partir das colunas
# `Pclass_1`, `Pclass_2`, e `Pclass_3` (qualitativas nominais)
df_train['Pclass'] = df_train['Pclass_1'] + 2*df_train['Pclass_2'] + 3*df_train['Pclass_3']
df_test['Pclass'] = df_test['Pclass_1'] + 2*df_test['Pclass_2'] + 3*df_test['Pclass_3']

# Para fins desse exercício, vou escolher três variáveis explicativas:
var_explicativas = [
    "Age",  # Quantitativa discreta
    "Pclass",  # Qualitativa ordinal
    "Sex",  # Qualitativa nominal
]

df_X = df_train[var_explicativas]
df_y = df_train["Survived"]
df_train = df_train[var_explicativas + ["Survived"]]
df_test = df_test[var_explicativas + ["Survived"]]

# 03. Construção do Algoritmo da Árvore

In [4]:
def calculate_gini(y):
    """
    Calcula a impureza Gini de um vetor de labels (Target).
    Exemplo de uso: calculate_gini(df_train['Survived'])
    """
    n = len(y)
    if n == 0:
        return 0
        
    gini = 1.0
    for i in y.unique():
        prob = np.sum(y == i) / n
        gini -= prob ** 2
        
    return gini

In [5]:
# Primeiro, vou criar uma função para abstrair o algoritmo CART
def find_cart_threshold(df, variable, target):
    # Primeiro passo é definir qual é o melhor threshold dessa variável.
    # Para isso vou ordenar o data frame por ela (para visualização), e 
    # calcular a quantidade de sobreviventes e casualidades para cada valor da
    # variável, e em seguida aplicar o mesmo cálculo de Gini e obter o melhor threshold
    possible_variable_values = df[variable].unique()

    if len(possible_variable_values) == 1:
        return None, 0

    gini_split_values = {}

    for value in possible_variable_values:
        lt_subset = df[df[variable] <= value]
        gt_subset = df[df[variable] > value]

        if len(lt_subset) == 0 or len(gt_subset) == 0:
            continue

        gini_lt = calculate_gini(lt_subset[target])
        gini_gt = calculate_gini(gt_subset[target])
        gini_split_values[value] = (gini_lt * len(lt_subset) + gini_gt * len(gt_subset)) / len(df)

    best_threshold = min(gini_split_values, key=gini_split_values.get)
    return best_threshold, gini_split_values[best_threshold]


# Função para abstrair o processo de encontrar o melhor split
def find_best_split(df, target, evaluated_features:float=1.0):
    """
    Entre todas as features, escolhe o (feature, threshold) com menor
    impureza Gini ponderada do split. Retorna (None, None, None) se não
    houver split válido.

    `evaluated_features` é um float entre 0 e 1 que indica a fração de features
    que serão avaliadas.
    """
    
    # Escolher aleatoriamente as features a serem avaliadas
    features = np.random.choice(df.columns, size=int(len(df.columns) * evaluated_features), replace=False)

    candidates = {}
    for var in features:
        if var == target:
            continue

        threshold, impurity = find_cart_threshold(df, var, target)
        if threshold is None:
            continue

        candidates[var] = (threshold, impurity)

    if not candidates:
        return None, None, None

    best_feature = min(candidates, key=lambda var: candidates[var][1])
    threshold, impurity = candidates[best_feature]
    return best_feature, threshold, impurity

In [6]:
# Construir classe que representa um nó na árvore
class Node:
    """Nó de uma árvore de decisão.

    `prediction` guarda a classe majoritária do nó (útil na pós-poda:
    ao colapsar uma subárvore, a folha herda essa predição).
    Folha = ausência de filhos (left/right is None).
    """

    def __init__(
        self,
        feature=None,
        threshold=None,
        left=None,
        right=None,
        prediction=None,
        n_samples=0,
    ):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.prediction = prediction
        self.n_samples = n_samples

    @property
    def is_leaf(self):
        return self.left is None and self.right is None

    def predict_row(self, row):
        """Percorre a árvore até uma folha para uma única observação (Series/dict)."""
        node = self
        while not node.is_leaf:
            if row[node.feature] <= node.threshold:
                node = node.left
            else:
                node = node.right
        return node.prediction

    def predict(self, df):
        """Predições (0/1) para um DataFrame. Retorna um ndarray alinhado às linhas."""
        return df.apply(lambda row: self.predict_row(row), axis=1).to_numpy()

    # Em algoritmos de bagging e RandomForest, não realizamos pós-pruning,
    # porque a variância do modelo é reduzida pela agregação de várias árvores.

def majority_class(y):
    """Determina a predição de uma folha."""
    return y.value_counts().idxmax()

def build_tree(
    df,
    target="Survived",
    depth=0,
    max_depth=None,
    min_samples_split=2,
    evaluated_features:float=1.0,
):
    """
    Constrói recursivamente uma árvore CART.

    Critérios de parada (folha):
      - nó puro (uma única classe)
      - menos de min_samples_split amostras
      - profundidade >= max_depth (se informado)
      - nenhum split válido restante

    Todo nó (interno ou folha) guarda `prediction` = classe majoritária
    e `n_samples`, necessários para pós-poda (REP / CCP).
    """
    y = df[target]
    maj = majority_class(y)
    n = len(df)

    pure = y.nunique() == 1
    too_small = n < min_samples_split
    too_deep = max_depth is not None and depth >= max_depth

    if pure or too_small or too_deep:
        return Node(prediction=maj, n_samples=n)

    feature, threshold, _ = find_best_split(df, target, evaluated_features)
    if feature is None:
        return Node(prediction=maj, n_samples=n)

    left_df = df[df[feature] <= threshold]
    right_df = df[df[feature] > threshold]

    # Split degenerado (não deve ocorrer se find_cart_threshold estiver ok)
    if len(left_df) == 0 or len(right_df) == 0:
        return Node(prediction=maj, n_samples=n)

    left = build_tree(
        left_df,
        target=target,
        depth=depth + 1,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        evaluated_features=evaluated_features
    )
    right = build_tree(
        right_df,
        target=target,
        depth=depth + 1,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        evaluated_features=evaluated_features
    )

    return Node(
        feature=feature,
        threshold=threshold,
        left=left,
        right=right,
        prediction=maj,
        n_samples=n,
    )


def print_tree(node, indent=0):
    """Impressão legível da árvore para inspeção."""
    pad = "  " * indent
    if node.is_leaf:
        print(f"{pad}→ predict {node.prediction} (n={node.n_samples})")
        return

    print(f"{pad}if {node.feature} <= {node.threshold}:  # n={node.n_samples}")
    print_tree(node.left, indent + 1)
    print(f"{pad}else:  # {node.feature} > {node.threshold}")
    print_tree(node.right, indent + 1)

# 04. Execução de estratégia de bagging

In [7]:
# Boostraping do dataset
n_samples = len(df_train)
n_trees = 10

trees = list()
for i in range(n_trees):
    # Um dataset `bootstrap` é criado a partir da seleção aleatória
    # **com reposição** (valores repetidos são possíveis) de amostras
    # do dataset original.
    bootstrap_indices = np.random.choice(n_samples, size=n_samples, replace=True)
    df_train_bootstrap = df_train.iloc[bootstrap_indices]

    # Construir árvore do dataset bootstrap
    tree = build_tree(df_train_bootstrap, max_depth=3, evaluated_features=1.0)
    trees.append(tree)

## 04.01. Avaliação do modelo

In [8]:
# Cálculo da acurácia de teste
y_true = df_test["Survived"].to_numpy()
y_pred = np.zeros(len(df_test))

for tree in trees:
    y_pred += tree.predict(df_test)

# Média das predições (voto suave). Para classificação, convertemos
# para classe com o limiar 0.5 — equivalente à moda quando n_trees é ímpar.
y_pred = (y_pred / n_trees) >= 0.5

accuracy = (y_pred == y_true).mean()
print(f"Acurácia do modelo: {accuracy:.2%}")

Acurácia do modelo: 80.00%


In [9]:
# Cálculo da acurácia de treinamento
y_true = df_train["Survived"].to_numpy()
y_pred = np.zeros(len(df_train))

for tree in trees:
    y_pred += tree.predict(df_train)

y_pred = (y_pred / n_trees) >= 0.5

accuracy = (y_pred == y_true).mean()
print(f"Acurácia do modelo: {accuracy:.2%}")

Acurácia do modelo: 80.93%


# 05. Execução de modelo RandomForest

In [10]:
# Boostraping do dataset
n_samples = len(df_train)
n_trees = 10

trees = list()
for i in range(n_trees):
    # Um dataset `bootstrap` é criado a partir da seleção aleatória
    # **com reposição** (valores repetidos são possíveis) de amostras
    # do dataset original.
    bootstrap_indices = np.random.choice(n_samples, size=n_samples, replace=True)
    df_train_bootstrap = df_train.iloc[bootstrap_indices]

    # Construir árvore do dataset bootstrap
    tree = build_tree(df_train_bootstrap, max_depth=3, evaluated_features=2/3)
    trees.append(tree)

## 05.01. Avaliação do modelo

In [11]:
# Cálculo da acurácia de teste
y_true = df_test["Survived"].to_numpy()
y_pred = np.zeros(len(df_test))

for tree in trees:
    y_pred += tree.predict(df_test)

# Média das predições (voto suave). Para classificação, convertemos
# para classe com o limiar 0.5 — equivalente à moda quando n_trees é ímpar.
y_pred = (y_pred / n_trees) >= 0.5

accuracy = (y_pred == y_true).mean()
print(f"Acurácia do modelo: {accuracy:.2%}")

Acurácia do modelo: 79.00%


In [12]:
# Cálculo da acurácia de treinamento
y_true = df_train["Survived"].to_numpy()
y_pred = np.zeros(len(df_train))

for tree in trees:
    y_pred += tree.predict(df_train)

y_pred = (y_pred / n_trees) >= 0.5

accuracy = (y_pred == y_true).mean()
print(f"Acurácia do modelo: {accuracy:.2%}")

Acurácia do modelo: 79.42%


# Anotações Extras

## O que este exercício estava testando

O resultado esperado da comparação entre `Bagging` e `Random Forest` era que `Random Forest` apresentasse uma variância menor, uma vez que as árvores de `Bagging` tendem a ser mais homogêneas, principalmente quando o dataset contém variáveis explicativas bastante dominantes.

Neste Titanic, essa variável dominante é `Sex`. No exercício da árvore simples, o stump de `Sex` já tinha o menor Gini. No bagging com `evaluated_features=1.0`, quase toda árvore tende a começar (e se parecer) com `Sex` → `Pclass`/`Age`. As árvores ficam correlacionadas; o voto adiciona pouco.

## Outros detalhes importantes

- **CART + Gini, split binário** (`≤` vs `>`): o mesmo motor da árvore simples; bagging/RF só mudam *em quais dados* e *quais colunas* esse motor roda.
- **Bootstrap:** cada árvore vê ~63,2% das linhas únicas; o restante é *out-of-bag* (OOB). Dá para estimar erro sem `df_test`, predizendo cada linha só com as árvores que **não** a usaram.
- **Sem pós-poda:** a regularização é o voto + (na RF) o sorteio de features. `max_depth` aqui é pré-poda extra, pedagógica, mas mascara parte do efeito RF.
- **Voto:** a média `≥ 0.5` é voto suave. Em classificação o padrão é a **moda** (voto duro). Com $B$ par, `≥ 0.5` em empate $5 \times 5$ cai para a classe $1$; isso é uma escolha, não um teorema.